## I created this notebook which includes all the neccessary knowledge about Linear Regression

Sections covered
- Gradient descent optimization: implementing manual batch gradient descent to iteratively minimize the Mean Squared Error cost function
- Overfitting
- Ridge regularization: instead of optimizing the MSE cost function, I'll exted it with a penalty term to reduce the risk of overfitting. Cost function + penalty term will make the objective function

Cost Function (J): Measures prediction error via Mean Squared Error:

J = SUM(y_pred - y_target)^2 where y_pred = X @ w

During each epoch (iteration), weights are updated proportionally to the partial derivative of the cost function with respect to each specific weight

In [114]:
import numpy as np

In [106]:
X = np.array([[10,2,1],[12,2,2],[15,3,1],[16,3,3],[18,3,2],[20,4,1]], dtype=float)
y_target = np.array([200, 220, 280, 270, 320, 380], dtype=float)
n = len(y_target)

In [107]:
def MSE(y_target: np.array, y_pred: np.array) -> float:
    return np.mean((y_target - y_pred)**2)

y_pred = np.array([0,0,0,0,0,0]) 
MSE(y_target, y_pred)

81083.33333333333

In [108]:
X = np.hstack((np.ones((n,1)),X))
X

array([[ 1., 10.,  2.,  1.],
       [ 1., 12.,  2.,  2.],
       [ 1., 15.,  3.,  1.],
       [ 1., 16.,  3.,  3.],
       [ 1., 18.,  3.,  2.],
       [ 1., 20.,  4.,  1.]])

In [120]:
def fit_linear_regression(X: np.array, y_target: np.array, lr: float, epoch: int,) -> tuple:
  cost_history = []
  weights = np.zeros(X.shape[1])
  n = len(y_target)
  
  for _ in range(epoch):
    y_pred = X @ weights 
    for i in range(len(weights)):
      gradient = 0
      for j in range(n):
        gradient += (y_pred[j] - y_target[j]) * X[j, i]
      mse_grad = gradient / n

      total_gradient = mse_grad
      weights[i] -= lr * total_gradient

    y_pred = X @ weights
    mse_loss = np.mean((y_target - y_pred) ** 2)
    cost_history.append(mse_loss)

  return weights, cost_history


weights, cost_history = fit_linear_regression(X, y_target, lr=0.001, epoch=20)
print("Trained Weights:", weights)
print("Cost History:", cost_history)

Trained Weights: [ 1.10451613 17.38848091  3.28583659  1.65493883]
Cost History: [45264.47490324074, 25309.13930319618, 14191.625161940894, 7997.802289305769, 4547.046383682662, 2624.4968977811227, 1553.335661500536, 956.4964868973425, 623.909791902066, 438.5424647057491, 335.19349363850256, 277.53835597407436, 245.34008113014025, 227.3243301826346, 217.20998912600155, 211.4977491251087, 208.2380893341217, 206.3449004924291, 205.2130936923808, 204.50556413029585]


Showcasing how multicolleration can unstabelize weights:

In [122]:
def make_poly_features(x_array, degree):
    features = []
    for d in range(1, degree + 1):
        features.append(x_array**d)
    return np.column_stack(features)


# DATASET SETUP & FEATURE ENGINEERING
X_train_raw = np.array([10, 12, 15, 16, 18, 20])
y_train     = np.array([200, 220, 280, 270, 320, 380])

X_test_raw  = np.array([14, 22])
y_test      = np.array([250, 400])

X_train = make_poly_features(X_train_raw, 5)
X_test  = make_poly_features(X_test_raw, 5)

X_train_mean = np.mean(X_train, axis=0)
X_train_std  = np.std(X_train, axis=0)

X_train = (X_train - X_train_mean) / X_train_std
X_test  = (X_test - X_train_mean) / X_train_std 

X_train_final = np.column_stack([np.ones(len(X_train)), X_train])
X_test_final  = np.column_stack([np.ones(len(X_test)), X_test])

n = X_train_final.shape[0]

weights, train_cost_history = fit_linear_regression(X_train_final, y_train, 0.01, 40000)

train_preds = X_train_final @ weights
test_preds  = X_test_final @ weights

train_mse = MSE(y_train, train_preds)
test_mse  = MSE(y_test, test_preds)

print(f"Final Weights:\n{weights}\n")
print(f"Training MSE (Near 0 - Memorized!): {train_mse:.2f}")
print(f"Testing MSE (Exploded! - Fails on new data): {test_mse:.2f}")

Final Weights:
[278.33333333  23.53831954   7.07297929   1.44176466   6.73988414
  21.52529599]

Training MSE (Near 0 - Memorized!): 67.80
Testing MSE (Exploded! - Fails on new data): 1165.79


The Model Overfit

Both the analytical (closed-form) solution and iterative gradient descent can fail and severely overfit when input features are highly correlated (collinear):

- Closed-Form Solution Failure XT * X^-1:
  Collinear columns make the matrix non-invertible (singular). Attempting to invert it is mathematically similar to dividing by zero, causing weight values to explode to infinity.
  
- Gradient Descent Failure: Because gradient descent focuses solely on minimizing training loss, it exploits correlated features by letting massive weights counterweight each other. This creates wild, unstable value that memorize the training data while failing on unseen data.

## Regularization

- Preventing Overfitting
By adding a cost for large weights, regularization stops models from memorizing noise in the training set. Makes the model generalizable.

- Automated Feature Selection / Shrinkage
Regularization automatically reduces the influence of less important features so they don't hijack predictions.

- Guarding Against Noisy Data 
Without regularization, models trained on noisy data develop extreme sensitivity (tiny change in an input feature drastically changes the output). Regularization ensures the model remains robust against minor data fluctuations.

In [123]:
def fit_linear_regression_ridge(X: np.array, y_target: np.array, lr: float, epoch: int, alpha: float) -> np.array: 
    cost_history = []
    weights = np.zeros(X.shape[1])
    y_pred = X @ weights
    for _ in range(epoch):
        for i in range(len(weights)):
            gradient = 0
            for j in range(n):
                
                gradient += ((y_pred[j] - y_target[j]) * X[j,i])  
            if i != 0:
                weights[i] -= (( gradient/n + (2 * alpha * weights[i])) * lr)
            else:
                weights[i] -= (( gradient/n) * lr)
        y_pred = X @ weights
        cost_history.append(MSE(y_target, X @ weights))
    return weights, cost_history

weights_ridge, cost_history_ridge = fit_linear_regression_ridge(X, y_target, 0.001, 20, 0.2)
print(cost_history_ridge)
weights, cost_history = fit_linear_regression(X, y_target, 0.001, 20)
print(cost_history)

[45264.47490324074, 25318.246762950872, 14208.571627522033, 8018.936487841419, 4569.131863468506, 2645.3870953701394, 1571.886439728294, 972.2804372300114, 636.940858470792, 449.06954506903895, 343.56468513047275, 284.120274519, 250.47495491974996, 231.31085307759747, 220.29800218035462, 213.8902689636568, 210.09671911270826, 207.79634817154582, 206.35558988272132, 205.41461230728927]
[45264.47490324074, 25309.13930319618, 14191.625161940894, 7997.802289305769, 4547.046383682662, 2624.4968977811227, 1553.335661500536, 956.4964868973425, 623.909791902066, 438.5424647057491, 335.19349363850256, 277.53835597407436, 245.34008113014025, 227.3243301826346, 217.20998912600155, 211.4977491251087, 208.2380893341217, 206.3449004924291, 205.2130936923808, 204.50556413029585]


standardize!

In [135]:
X_train_raw = np.array([10, 12, 15, 16, 18, 20])
y_train = np.array([200, 220, 280, 270, 320, 380])

X_test_raw = np.array([14, 22])
y_test = np.array([250, 400])


def make_poly_features(x_array, degree):
  features = []
  for d in range(1, degree + 1):
    features.append(x_array**d)
  return np.column_stack(features)


X_train = make_poly_features(X_train_raw, 5)
X_test = make_poly_features(X_test_raw, 5)

mean = np.mean(X_train, axis=0)
std = np.std(X_train, axis=0)
X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std  

X_train_final = np.column_stack([np.ones(len(X_train_scaled)), X_train_scaled])
X_test_final = np.column_stack([np.ones(len(X_test_scaled)), X_test_scaled])


# TEST A: Unregularized
weights_unreg, _ = fit_linear_regression(X_train_final, y_train, 0.01, 30000)
test_preds_unreg = X_test_final @ weights_unreg
test_mse_unreg = np.mean((test_preds_unreg - y_test) ** 2)

print("--- UNREGULATED MODEL ---")
print(f"Weights: {weights_unreg}")
print(f"Test MSE: {test_mse_unreg:.2f}\n")


# TEST B: Regularized with Ridge 
weights_ridge, _ = fit_linear_regression_ridge(X_train_final, y_train, 0.01, 30000, 1.0)
test_preds_ridge = X_test_final @ weights_ridge
test_mse_ridge = np.mean((test_preds_ridge - y_test) ** 2)

print("--- RIDGE REGULARIZED MODEL (alpha=1.0) ---")
print(f"Weights: {weights_ridge}")
print(f"Test MSE: {test_mse_ridge:.2f}")

--- UNREGULATED MODEL ---
Weights: [278.33333333  21.12186085   8.42696148   3.97402309   7.82949299
  18.89489635]
Test MSE: 1128.13

--- RIDGE REGULARIZED MODEL (alpha=1.0) ---
Weights: [278.33333333   8.52560374   8.59044613   8.58681378   8.5292741
   8.4361478 ]
Test MSE: 22.78
